# aufbau — Semantic Prefix Grammar experiments

This notebook demonstrates the aufbau Python API: grammar inspection, tokenization, parsing, AST traversal, and type resolution.

In [1]:
import aufbau
print(aufbau.version())

0.1.2


## 1. Grammar loading and inspection

Load a grammar from `.auf` source text. Inspect nonterminals, productions, rules, and tokenize input.

In [3]:
STLC = r"""
    Identifier ::= /[a-z]+/
    Type ::= 'A' | 'B' | 'C' | /[A-Z][a-zA-Z0-9]*/
    Variable(var) ::= Identifier[x]
    Lambda(lambda) ::= 'λ' Identifier[param] ':' Type[τ] '.' Expr[body]
    Application(app) ::= Expr[func] Expr[arg]
    Expr ::= Variable | Lambda | Application | '(' Expr ')'

    x ∈ Γ
    ----------- (var)
    Γ(x)

    Γ[param:τ] ⊢ body : ?T
    ----------- (lambda)
    τ → ?T

    Γ ⊢ func : τ₂ → ?T, Γ ⊢ arg : τ₂
    ----------- (app)
    ?T
"""

g = aufbau.SPG(STLC)
print(f"start: {g.start}")
print(g)


start: Expr


### Inspect productions

In [ ]:
for nt in g.nonterminals():
    rule = g.rule_for(nt) or "(none)"
    prods = g.productions(nt)
    print(f"{nt} [{rule}]")
    for p in prods:
        symbols = [f"{s.kind}[{s.name}]" + (f"{{{s.binding}}}" if s.binding else "") for s in p.rhs]
        print(f"  -> {' '.join(symbols)}")
    print()

### Tokenization

In [ ]:
segs = g.tokenize("λx:A.λy:B.x y")
for s in segs:
    print(f"  [{s.index}] '{s.text}' (bytes {s.start}..{s.end})")

## 2. Parsing and feeding

Create a `Synthesizer` with a grammar and input. Parse, feed tokens incrementally, and check completeness.

In [ ]:
# Arithmetic grammar
ARITH = r"""
    Number ::= /[0-9]+/
    Identifier ::= /[a-z][a-zA-Z0-9]*/
    Literal ::= Number
    Variable ::= Identifier
    Operator ::= '+' | '-' | '*' | '/'
    Primary ::= Literal | Variable | '(' Expression ')'
    Expression ::= Primary | Primary Operator Expression
"""

s = aufbau.Synthesizer(ARITH, "1 + 2 * 3")
print(f"complete: {s.is_complete()}")
print(s.parse())

In [ ]:
# Incremental feeding
s = aufbau.Synthesizer(ARITH, "")
for token in ["1", "+", "2"]:
    result = s.feed(token)
    print(f"feed('{token}') -> complete={s.is_complete()} input='{s.input()}'")
    print(f"  tree: {result[:80]}...")

### try_feed — speculative feeding without state change

In [ ]:
s = aufbau.Synthesizer(ARITH, "1")
print(f"current: '{s.input()}'")
result = s.try_feed("+")
print(f"try_feed('+') -> ok={result is not None}")
print(f"state unchanged: '{s.input()}'")
result = s.try_feed(")")
print(f"try_feed(')') -> ok={result is not None}")

## 3. Typed parsing with context

Add type bindings to the context, then parse typed programs.

In [ ]:
s = aufbau.Synthesizer(STLC, "λx:A.x")
ast = s.ast()
print(f"nodes: {ast.node_count()}, complete: {ast.is_complete()}")
for root in ast.roots:
    ty = ast.type_of(root.evidence)
    print(f"  root evidence={root.evidence} type={ty}")

### Adding context bindings

In [ ]:
s = aufbau.Synthesizer(STLC, "f x")
s.add_to_ctx("f", "A -> B")
s.add_to_ctx("x", "A")
ast = s.ast()
print(f"complete: {ast.is_complete()}")
for root in ast.roots:
    ty = ast.type_of(root.evidence)
    print(f"  type: {ty}")

## 4. AST traversal

Walk the parse tree: nodes, children, terminals, and evidence.

In [ ]:
def walk(node, depth=0):
    indent = "  " * depth
    ty = ast.type_of(node.evidence) if node.evidence else "?"
    print(f"{indent}Node({node.nodeid}) nt={node.nt_name} evidence={node.evidence} type={ty} complete={node.is_complete}")
    for child in node.children:
        if child.kind == "node":
            walk(child.node, depth + 1)
        else:
            print(f"{indent}  term '{child.terminal_text}' complete={child.terminal_complete}")

s = aufbau.Synthesizer(STLC, "λx:A.x")
ast = s.ast()
for root in ast.roots:
    walk(root)

## 5. Inspecting rules

Get typing rules by name and inspect their premises and bindings.

In [ ]:
s = aufbau.Synthesizer(STLC, "")

for name in s.grammar().rule_names():
    rule = s.get_rule(name)
    if rule:
        print(f"[{rule.name}] premises={rule.premises} bindings={rule.bindings()}")
        print(rule.pretty(2))
        print()

## 6. Regex engine

Test the regex derivative engine for pattern matching and prefix recognition.

In [ ]:
r = aufbau.Regex("a*b")
print(f"pattern: {r.pattern()}")
print(f"matches 'ab':   {r.matches('ab')}")
print(f"matches 'aab':  {r.matches('aab')}")
print(f"matches 'a':    {r.matches('a')}")
print(f"nullable:       {r.is_nullable()}")
print()

status = r.prefix_match("aa")
print(f"prefix 'aa': complete={status.is_complete()} prefix={status.is_prefix()} extensible={status.is_extensible()} nomatch={status.is_no_match()}")

In [ ]:
# Derivatives
r = aufbau.Regex("abc")
d = r.derivative("ab")
print(f"deriv('a') on 'abc' -> pattern={d.pattern()}, matches 'c'={d.matches('c')}")
print(f"match_len('a')={r.match_len('a')}, match_len('ab')={r.match_len('ab')}")